In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast  # Mixed Precision for faster training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc, average_precision_score
from sklearn.cluster import KMeans
from scipy.stats import beta as beta_dist
import joblib
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# TPU/GPU DETECTION (PyTorch)
# =============================================================================
print("\n" + "="*80)
print("HARDWARE DETECTION")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"Platform: {platform.platform()}")

# TPU setup
TPU_AVAILABLE = False
xm = None
xmp = None

try:
    import torch_xla
    import torch_xla.core.xla_model as xla_model
    TPU_AVAILABLE = True
    xm = xla_model
    print(f"✅ TPU Available: torch_xla {torch_xla.__version__}")
    try:
        import torch_xla.distributed.xla_multiprocessing as xla_mp
        xmp = xla_mp
    except ImportError:
        pass
except ImportError:
    print("⚠️ TPU not available (torch_xla not installed)")
    print("   For Kaggle TPU: Runtime → Change runtime type → TPU")

# CUDA setup
if torch.cuda.is_available():
    print(f"\n✅ CUDA Available")
    print(f"   GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"   GPU {i}: {props.name} ({props.total_memory / 1e9:.2f} GB)")
else:
    print("\n⚠️ CUDA not available")

print("="*80 + "\n")

# Device selection
USE_XLA = False
DEVICE = None

if TPU_AVAILABLE and xm is not None:
    USE_XLA = True
    DEVICE = xm.xla_device()
    print(f"🎯 Using TPU: {DEVICE}")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🎯 Using CUDA: {DEVICE}")
else:
    DEVICE = torch.device("cpu")
    print(f"🎯 Using CPU: {DEVICE}")

print("")

# =============================================================================
# CONFIGURATION
# =============================================================================

# Dataset
DATA_DIR = "/kaggle/input/cic-ids2017/MachineLearningCVE"
CSV_FILES = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
]

# Output
OUT_DIR = "/kaggle/working/models"
PLOT_DIR = "/kaggle/working/plots"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Set seed for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Training hyperparameters
EPOCHS = 100
BATCH_SIZE = 4096 if torch.cuda.is_available() else (1024 if TPU_AVAILABLE else 512)
LEARNING_RATE = 0.001
PATIENCE = 10


# =============================================================================
# MODEL SELECTION - MorphIDS Deep Learning Pool + Apollon
# =============================================================================
MODELS_TO_TRAIN = {
    'mlp_model': True,          # ✅ Deep MLP (MorphIDS)
    'cnn_model': True,          # ✅ 1D CNN (MorphIDS)
    'tcn_model': True,          # ✅ Temporal CNN (MorphIDS)
    'bilstm_model': True,       # ✅ BiLSTM-Attention (MorphIDS)
    'apollon_model': True,      # ✅ Apollon (MAB with MorphIDS Pool)
}


# =============================================================================
# MORPHIDS MODEL ARCHITECTURES (Deep Learning)
# =============================================================================

class MLPDropout(nn.Module):
    """Deep MLP with dropout for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, d_out)
        )
    def forward(self, x): return self.net(x)


class DeepCNN(nn.Module):
    """1D CNN for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.proj = nn.Linear(d_in, 64)
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, d_out))
    def forward(self, x):
        x = F.relu(self.proj(x)).unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return self.fc(self.pool(x).squeeze(-1))


class DeepTCN(nn.Module):
    """Temporal Convolutional Network for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.input = nn.Linear(d_in, 64)
        self.tcn1 = nn.Conv1d(1, 16, kernel_size=3, padding=2, dilation=2)
        self.tcn2 = nn.Conv1d(16, 32, kernel_size=3, padding=4, dilation=4)
        self.tcn3 = nn.Conv1d(32, 32, kernel_size=3, padding=8, dilation=8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, d_out))
    def forward(self, x):
        x = F.relu(self.input(x)).unsqueeze(1)
        x = F.relu(self.tcn1(x))
        x = F.relu(self.tcn2(x))
        x = F.relu(self.tcn3(x))
        return self.fc(self.pool(x).squeeze(-1))


class BiLSTMAttention(nn.Module):
    """Bidirectional LSTM with attention mechanism for IDS classification."""
    def __init__(self, d_in: int, n_classes: int = 2, p: float = 0.2, hidden_size: int = 128, num_layers: int = 2, num_heads: int = 4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.proj_size = 64
        
        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(d_in, self.proj_size),
            nn.LayerNorm(self.proj_size),
            nn.ReLU(),
            nn.Dropout(p)
        )
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=self.proj_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=p if num_layers > 1 else 0
        )
        
        # Multi-head attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size * 2,
            num_heads=num_heads,
            dropout=p,
            batch_first=True
        )
        
        # Output layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, n_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.size(0)
        
        # Project input
        x = self.input_proj(x)
        x = x.unsqueeze(1)  # (batch, 1, proj_size)
        
        # LSTM forward pass
        lstm_out, _ = self.lstm(x)  # (batch, 1, hidden*2)
        
        # Self-attention
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        
        # Global average pooling
        pooled = attn_out.mean(dim=1)  # (batch, hidden*2)
        
        # Classification
        logits = self.classifier(pooled)
        return logits


# =============================================================================
# PYTORCH MODEL WRAPPER (for Apollon compatibility)
# =============================================================================

class PyTorchModelWrapper:
    """Wrapper to make PyTorch models compatible with sklearn-like interface."""
    
    def __init__(self, model_class, d_in, d_out=2, device=None, epochs=50, batch_size=1024, lr=0.001):
        self.model_class = model_class
        self.d_in = d_in
        self.d_out = d_out
        self.device = device if device else DEVICE
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.model = None
        self.class_weights = None
        
    def fit(self, X, y):
        """Train the model."""
        # Convert to tensors
        X_tensor = torch.FloatTensor(X).to(self.device)
        y_tensor = torch.LongTensor(y).to(self.device)
        
        # Calculate class weights
        class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
        self.class_weights = torch.FloatTensor(class_weights).to(self.device)
        
        # Create model
        self.model = self.model_class(self.d_in, self.d_out).to(self.device)
        
        # Create dataloader
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        # Training setup
        criterion = nn.CrossEntropyLoss(weight=self.class_weights)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        
        # Train
        self.model.train()
        for epoch in range(self.epochs):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
        
        return self
    
    def predict(self, X):
        """Predict class labels with batched processing to avoid OOM."""
        self.model.eval()
        all_preds = []
        batch_size = self.batch_size
        
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch_X = torch.FloatTensor(X[i:i+batch_size]).to(self.device)
                outputs = self.model(batch_X)
                _, predicted = torch.max(outputs, 1)
                all_preds.append(predicted.cpu())
                del batch_X, outputs  # Free memory
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        
        return torch.cat(all_preds).numpy()
    
    def predict_proba(self, X):
        """Predict class probabilities with batched processing to avoid OOM."""
        self.model.eval()
        all_probs = []
        batch_size = self.batch_size
        
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                batch_X = torch.FloatTensor(X[i:i+batch_size]).to(self.device)
                outputs = self.model(batch_X)
                probs = F.softmax(outputs, dim=1)
                all_probs.append(probs.cpu())
                del batch_X, outputs  # Free memory
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        
        return torch.cat(all_probs).numpy()


# =============================================================================
# APOLLON ARCHITECTURE (Multi-Armed Bandit with MorphIDS Deep Learning Pool)
# =============================================================================

class MultiArmedBanditThompsonSampling:
    """
    Apollon with MorphIDS Deep Learning Pool
    
    This implementation uses Multi-Armed Bandits (MAB) with Thompson Sampling to dynamically
    select the optimal Deep Learning classifier for each input. Instead of traditional ML
    models, we use MorphIDS deep learning architectures (MLP, CNN, TCN, BiLSTM-Attention).
    
    Reference: Apollon paper - "Apollon: A Robust Defence System against Adversarial 
    Machine Learning Attacks in Intrusion Detection Systems"
    
    Modified to use MorphIDS pool for enhanced detection capabilities.
    """
    
    def __init__(self, d_in, n_arms=4, n_clusters=2, seed=42, device=None):
        """
        Initialize the Multi-Armed Bandit with Thompson Sampling using MorphIDS models.
        
        Parameters:
            d_in: Input dimension (number of features)
            n_arms: Number of classifier arms (default: 4 - MLP, CNN, TCN, BiLSTM)
            n_clusters: Number of clusters for input space partitioning
            seed: Random seed for reproducibility
            device: PyTorch device (CPU/CUDA/TPU)
        """
        self.d_in = d_in
        self.n_arms = n_arms
        self.n_clusters = n_clusters
        self.seed = seed
        self.device = device if device else DEVICE
        np.random.seed(seed)
        torch.manual_seed(seed)
        
        # Initialize MorphIDS Deep Learning arms
        self.arms = [
            PyTorchModelWrapper(MLPDropout, d_in, 2, self.device, epochs=30, batch_size=1024),
            PyTorchModelWrapper(DeepCNN, d_in, 2, self.device, epochs=30, batch_size=1024),
            PyTorchModelWrapper(DeepTCN, d_in, 2, self.device, epochs=30, batch_size=1024),
            PyTorchModelWrapper(BiLSTMAttention, d_in, 2, self.device, epochs=30, batch_size=1024),
        ]
        self.arm_names = ['MLP', 'CNN', 'TCN', 'BiLSTM-Attention']
        
        self.cluster_centers = None
        self.cluster_assignments = None
        self.reward_sums = {}
        for cluster in range(n_clusters):
            self.reward_sums[cluster] = np.zeros(n_arms)
        
        # Beta distribution parameters for Thompson Sampling
        self.alpha = np.ones(self.n_arms)
        self.beta = np.ones(self.n_arms)
        
        self.training_time = 0.0
    
    def train(self, X_train, y_train):
        """
        Train all MorphIDS classifier arms and cluster the input space.
        
        Parameters:
            X_train: Training features
            y_train: Training labels
        """
        start_time = time.time()
        
        # Cluster the training data
        print(f"    Clustering input space into {self.n_clusters} clusters...")
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.seed, n_init=10)
        self.cluster_assignments = kmeans.fit_predict(X_train)
        self.cluster_centers = kmeans.cluster_centers_
        
        # Print cluster distribution
        for i in range(self.n_clusters):
            cluster_count = np.sum(self.cluster_assignments == i)
            print(f"      Cluster {i}: {cluster_count:,} samples ({cluster_count/len(y_train)*100:.2f}%)")
        
        # Train all MorphIDS arms on the full training data
        print(f"\n    Training {self.n_arms} MorphIDS Deep Learning arms...")
        for arm_idx, (arm, name) in enumerate(zip(self.arms, self.arm_names)):
            print(f"      [{arm_idx+1}/{self.n_arms}] Training {name}...")
            arm.fit(X_train, y_train)
        
        # Calculate reward sums for each arm in each cluster
        print(f"\n    Computing arm rewards per cluster...")
        for cluster_idx in range(self.n_clusters):
            cluster_mask = self.cluster_assignments == cluster_idx
            cluster_X = X_train[cluster_mask]
            cluster_y = y_train[cluster_mask]
            
            for arm_idx, arm in enumerate(self.arms):
                if len(cluster_X) > 0:
                    arm_preds = arm.predict(cluster_X)
                    accuracy = np.mean(arm_preds == cluster_y)
                    self.reward_sums[cluster_idx][arm_idx] = accuracy
                    print(f"      Cluster {cluster_idx}, {self.arm_names[arm_idx]}: {accuracy*100:.2f}% accuracy")
        
        self.training_time = time.time() - start_time
        print(f"\n    ✅ Training completed in {self.training_time:.2f} seconds")
    
    def select_arm(self, cluster):
        """
        Select an arm using Thompson Sampling for the given cluster.
        
        Parameters:
            cluster: Cluster index
            
        Returns:
            Selected arm index
        """
        theta = np.zeros(self.n_arms)
        for arm_idx in range(self.n_arms):
            # Sample from Beta distribution
            alpha_param = self.alpha[arm_idx] + self.reward_sums[cluster][arm_idx]
            beta_param = self.beta[arm_idx] + 1 - self.reward_sums[cluster][arm_idx]
            theta[arm_idx] = np.random.beta(alpha_param, beta_param)
        return np.argmax(theta)
    
    def predict(self, X_test):
        """
        Predict using Thompson Sampling to dynamically select classifiers.
        
        Parameters:
            X_test: Test features
            
        Returns:
            predictions: Predicted labels
            selected_arms: Arms selected for each sample
        """
        # Assign each sample to a cluster
        clusters = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            distances = np.linalg.norm(self.cluster_centers - X_test[i], axis=1)
            clusters[i] = np.argmin(distances)
        
        # Select arm for each sample using Thompson Sampling
        selected_arms = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            selected_arms[i] = self.select_arm(clusters[i])
        
        # Predict using selected arms
        predictions = np.zeros(len(X_test), dtype=int)
        for arm_idx in range(self.n_arms):
            arm_mask = selected_arms == arm_idx
            if np.sum(arm_mask) > 0:
                arm_X = X_test[arm_mask]
                predictions[arm_mask] = self.arms[arm_idx].predict(arm_X)
        
        return predictions, selected_arms
    
    def predict_proba(self, X_test):
        """
        Predict probabilities using Thompson Sampling.
        
        Parameters:
            X_test: Test features
            
        Returns:
            probabilities: Probability of positive class (Attack)
        """
        predictions, selected_arms = self.predict(X_test)
        
        # Get probabilities from each arm
        probabilities = np.zeros(len(X_test))
        for arm_idx in range(self.n_arms):
            arm_mask = selected_arms == arm_idx
            if np.sum(arm_mask) > 0:
                arm_X = X_test[arm_mask]
                if hasattr(self.arms[arm_idx], 'predict_proba'):
                    probs = self.arms[arm_idx].predict_proba(arm_X)
                    if probs.shape[1] > 1:
                        probabilities[arm_mask] = probs[:, 1]
                    else:
                        probabilities[arm_mask] = probs[:, 0]
                else:
                    # For classifiers without predict_proba, use predictions
                    probabilities[arm_mask] = predictions[arm_mask]
        
        return probabilities

# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================
print("\n" + "="*80)
print("DATA LOADING AND PREPROCESSING")
print("="*80)

# Metadata columns to drop
drop_columns = [
    "Flow ID",    
    'Fwd Header Length.1',
    "Source IP", "Src IP",
    "Source Port", "Src Port",
    "Destination IP", "Dst IP",
    "Destination Port", "Dst Port",
    "Timestamp",
]

# Load CSV files
print(f"Loading {len(CSV_FILES)} CSV files from {DATA_DIR}...")
individual_dfs = []
for idx, csv_file in enumerate(CSV_FILES, 1):
    file_path = os.path.join(DATA_DIR, csv_file)
    print(f"  [{idx}/{len(CSV_FILES)}] Loading: {csv_file}")
    df = pd.read_csv(file_path, sep=',', encoding='utf-8')
    print(f"      Initial shape: {df.shape}")
    print(f"      Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    individual_dfs.append(df)

print(f"\nTotal DataFrames loaded: {len(individual_dfs)}")
print(f"Total memory before optimization: {sum(df.memory_usage(deep=True).sum() for df in individual_dfs) / 1024**2:.2f} MB")

# Preprocess each dataframe
print("\nPreprocessing individual DataFrames...")
print("-" * 80)
for i, df in enumerate(individual_dfs):
    print(f"\nDataFrame {i} ({CSV_FILES[i]}):")
    
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()
    print(f"  Total columns: {len(df.columns)}")
    
    # Drop metadata columns
    dropped = [col for col in drop_columns if col in df.columns]
    df.drop(columns=drop_columns, inplace=True, errors='ignore')
    if dropped:
        print(f"  Dropped metadata columns: {len(dropped)}")
    
    # Standardize label naming
    df['Label'] = df['Label'].replace({'BENIGN': 'Benign'})
    
    # Replace infinite values with NaN
    inf_count = np.isinf(df.select_dtypes(include=[np.number]).values).sum()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    if inf_count > 0:
        print(f"  Infinite values replaced with NaN: {inf_count}")
    
    # Drop NaN values
    nan_rows = df.isna().any(axis=1).sum()
    df.dropna(inplace=True)
    if nan_rows > 0:
        print(f"  Rows with NaN removed: {nan_rows}")
    
    # Drop duplicates
    duplicates = df.duplicated().sum()
    df.drop_duplicates(inplace=True)
    if duplicates > 0:
        print(f"  Duplicate rows removed: {duplicates}")
    
    df.reset_index(drop=True, inplace=True)
    print(f"  Final shape: {df.shape}")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Concatenate all dataframes
print("\nConcatenating DataFrames...")
df_data = pd.concat(individual_dfs, axis=0, ignore_index=True)
print(f"Combined dataset shape: {df_data.shape}")

# Additional cleaning
print("\nFinal data cleaning...")
null_counts = df_data.isnull().sum()
print(f"  Null entries found: {null_counts.sum()}")
if null_counts.sum() > 0:
    df_data.dropna(inplace=True)
    print(f"  After removing nulls: {df_data.shape}")

duplicate_count = df_data.duplicated().sum()
print(f"  Duplicate entries found: {duplicate_count}")
if duplicate_count > 0:
    df_data.drop_duplicates(inplace=True)
    print(f"  After removing duplicates: {df_data.shape}")

df_data.reset_index(drop=True, inplace=True)

# Inspect label distribution
print("\n" + "-"*80)
print("LABEL DISTRIBUTION (Before Binarization)")
print("-"*80)
print("\nAbsolute counts:")
print(df_data['Label'].value_counts().to_string())
print(f"\nRelative proportions:")
for label, prop in df_data['Label'].value_counts(normalize=True).items():
    print(f"  {label:<30} {prop*100:>6.2f}%")
print(f"\nTotal unique labels: {df_data['Label'].nunique()}")
print(f"Total samples: {len(df_data):,}")

# Extract features and target
print("\n" + "-"*80)
print("FEATURE EXTRACTION")
print("-"*80)
X = df_data.drop('Label', axis=1).copy()
y = df_data['Label'].copy()

# Binarize labels: Benign=0, Attack=1
print("\nBinarizing labels (Benign=0, Attack=1)...")
y = y.map({'Benign': 0}).fillna(1).astype(int)
print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"Features data type: {X.dtypes.value_counts().to_dict()}")
print(f"\nBinary class distribution:")
for class_label, count in pd.Series(y).value_counts().sort_index().items():
    class_name = "Benign" if class_label == 0 else "Attack"
    print(f"  Class {class_label} ({class_name}): {count:>10,} samples ({count/len(y)*100:>5.2f}%)")
print(f"\nClass imbalance ratio: {pd.Series(y).value_counts().max() / pd.Series(y).value_counts().min():.2f}:1")

# =============================================================================
# DATA SPLITTING
# =============================================================================
print("\n" + "="*80)
print("DATA SPLITTING")
print("="*80)

# Split: 75% train, 10% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    stratify=y,
    test_size=0.25,
    random_state=seed,
    shuffle=True
)

# Split temp into val and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    stratify=y_temp,
    test_size=0.6,  # 0.6 of 25% = 15% test
    random_state=seed,
    shuffle=True
)

print(f"\nDataset split results:")
print(f"  Train set: {X_train.shape[0]:>10,} samples ({X_train.shape[0]/len(X)*100:>5.2f}%)")
print(f"  Val set:   {X_val.shape[0]:>10,} samples ({X_val.shape[0]/len(X)*100:>5.2f}%)")
print(f"  Test set:  {X_test.shape[0]:>10,} samples ({X_test.shape[0]/len(X)*100:>5.2f}%)")
print(f"\nClass distribution in splits:")
print(f"  Train - Benign: {(y_train==0).sum():>10,}, Attack: {(y_train==1).sum():>10,}")
print(f"  Val   - Benign: {(y_val==0).sum():>10,}, Attack: {(y_val==1).sum():>10,}")
print(f"  Test  - Benign: {(y_test==0).sum():>10,}, Attack: {(y_test==1).sum():>10,}")

# =============================================================================
# DATA SCALING
# =============================================================================
print("\n" + "="*80)
print("DATA SCALING (Z-SCORE NORMALIZATION)")
print("="*80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train set scaled: {X_train_scaled.shape}")
print(f"Val set scaled:   {X_val_scaled.shape}")
print(f"Test set scaled:  {X_test_scaled.shape}")

# Save scaler for inference
scaler_path = os.path.join(OUT_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"💾 Scaler saved: {scaler_path}")

# Convert labels to numpy arrays
y_train_np = y_train.values
y_val_np = y_val.values
y_test_np = y_test.values

# =============================================================================
# VISUALIZATION UTILITIES
# =============================================================================

def plot_confusion_matrix(cm, model_name, save_path):
    """Plot confusion matrix heatmap."""
    plt.figure(figsize=(8, 6))
    
    # Normalize confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Create annotations with both counts and percentages
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"
    
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', 
                xticklabels=['Benign', 'Attack'],
                yticklabels=['Benign', 'Attack'],
                cbar_kws={'label': 'Proportion'},
                linewidths=1, linecolor='gray')
    
    plt.title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    📊 Confusion matrix plot saved: {save_path}")
    plt.close()


def plot_roc_curve(labels, probs, model_name, save_path):
    """Plot ROC curve and save with AUC annotation."""
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc*100:.2f}%)')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'{model_name} - ROC Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    🔍 ROC curve plot saved: {save_path} (AUC={roc_auc*100:.2f}%)")
    plt.close()


def plot_pr_curve(labels, probs, model_name, save_path):
    """Plot Precision-Recall curve and save with Average Precision (AP) annotation."""
    precision_vals, recall_vals, _ = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)

    plt.figure(figsize=(8, 6))
    plt.plot(recall_vals, precision_vals, color='purple', lw=2, label=f'PR curve (AP = {ap*100:.2f}%)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'{model_name} - Precision-Recall Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    🔍 PR curve plot saved: {save_path} (AP={ap*100:.2f}%)")
    plt.close()


def plot_metrics_comparison(results, save_path):
    """Plot comparison of all models across different metrics."""
    metrics = ['accuracy', 'recall', 'f1', 'auc']
    metric_names = ['Accuracy', 'Detection Rate', 'F1 Score', 'AUC-ROC']
    model_names = list(results.keys())
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x = np.arange(len(metrics))
    width = 0.8 / len(model_names)
    
    # Extended color palette for 6 models
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    for i, model_name in enumerate(model_names):
        color = colors[i % len(colors)]
        values = [results[model_name][metric] * 100 for metric in metrics]
        offset = (i - len(model_names)/2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=model_name, color=color, alpha=0.8)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%',
                   ha='center', va='bottom', fontsize=9)
    
    ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 105])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n📊 Model comparison plot saved: {save_path}")
    plt.close()


def plot_model_summary(results, save_path):
    """Create a comprehensive summary visualization."""
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    model_names = list(results.keys())
    # Extended color palette for 6 models
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    model_colors = [colors[i % len(colors)] for i in range(len(model_names))]
    
    # 1. F1 Score comparison (main metric)
    ax1 = fig.add_subplot(gs[0, :])
    f1_scores = [results[m]['f1'] * 100 for m in model_names]
    bars = ax1.barh(model_names, f1_scores, color=model_colors, alpha=0.8)
    for i, (bar, score) in enumerate(zip(bars, f1_scores)):
        ax1.text(score + 0.5, i, f'{score:.2f}%', va='center', fontsize=11, fontweight='bold')
    ax1.set_xlabel('F1 Score (%)', fontsize=12, fontweight='bold')
    ax1.set_title('F1 Score Comparison (Primary Metric)', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.set_xlim([0, 105])
    
    # 2-5. Individual metric plots
    metrics = [('accuracy', 'Accuracy'), ('recall', 'Detection Rate'), 
               ('f1', 'F1 Score'), ('auc', 'AUC-ROC')]
    positions = [(1, 0), (1, 1), (2, 0), (2, 1)]
    
    for (metric, title), pos in zip(metrics, positions):
        ax = fig.add_subplot(gs[pos])
        values = [results[m][metric] * 100 for m in model_names]
        bars = ax.bar(range(len(model_names)), values, color=model_colors, alpha=0.8)
        
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
        
        ax.set_xticks(range(len(model_names)))
        ax.set_xticklabels(model_names, rotation=45, ha='right')
        ax.set_ylabel('Score (%)', fontsize=10)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim([0, 105])
    
    plt.suptitle('Model Performance Summary', fontsize=16, fontweight='bold', y=0.995)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"📊 Model summary plot saved: {save_path}")
    plt.close()


# =============================================================================
# TRAINING AND EVALUATION UTILITIES (PyTorch)
# =============================================================================

def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    """Train for one epoch with optional Mixed Precision."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        if not USE_XLA:
            inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        if scaler is not None:  # Mixed precision training
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        total += labels.size(0)
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
    
    if USE_XLA and xm is not None:
        xm.mark_step()
    
    epoch_loss = running_loss / max(1, total)
    epoch_acc = correct / max(1, total)
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            if not USE_XLA:
                inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            total += labels.size(0)
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
    
    if USE_XLA and xm is not None:
        xm.mark_step()
    
    val_loss = running_loss / total
    val_acc = correct / total
    return val_loss, val_acc


def evaluate_model(model, loader, device):
    """Comprehensive evaluation."""
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            if not USE_XLA:
                inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    if USE_XLA and xm is not None:
        xm.mark_step()
    
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc_val = roc_auc_score(all_labels, all_probs)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'detection_rate': recall,
        'f1': f1,
        'auc': auc_val,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels
    }


def train_pytorch_model(model_name, model, train_loader, val_loader, device, class_weights_tensor, epochs=EPOCHS):
    """Complete training loop with early stopping."""
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*80}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Device: {device} {'(TPU)' if USE_XLA else ''}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Early stopping patience: {PATIENCE} epochs")
    print("-" * 80)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    
    best_val_acc = 0.0
    patience_counter = 0
    best_model_state = None
    
    train_history = {'loss': [], 'acc': []}
    val_history = {'loss': [], 'acc': []}
    
    scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
    
    start_time = time.time()
    
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        train_history['loss'].append(train_loss)
        train_history['acc'].append(train_acc)
        val_history['loss'].append(val_loss)
        val_history['acc'].append(val_acc)
        
        scheduler.step(val_loss)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"    Epoch {epoch+1:>3}/{epochs} - Loss: {train_loss:.4f} - Acc: {train_acc*100:.2f}% - Val Loss: {val_loss:.4f} - Val Acc: {val_acc*100:.2f}% ✓")
        else:
            patience_counter += 1
            print(f"    Epoch {epoch+1:>3}/{epochs} - Loss: {train_loss:.4f} - Acc: {train_acc*100:.2f}% - Val Loss: {val_loss:.4f} - Val Acc: {val_acc*100:.2f}%")
        
        if patience_counter >= PATIENCE:
            print(f"\n    ⚠️ Early stopping at epoch {epoch+1}")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    elapsed = time.time() - start_time
    print(f"\n    ✅ Training completed in {elapsed:.2f} seconds")
    print(f"    Best validation accuracy: {best_val_acc*100:.2f}%")
    
    return model, train_history, val_history, elapsed


def train_apollon_model(X_train, y_train, X_test, y_test, d_in, n_clusters=2, save_path=None):
    """
    Train and evaluate the Apollon MAB model with MorphIDS pool.
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: Apollon (MAB with MorphIDS Deep Learning Pool)")
    print(f"{'='*80}")
    print(f"Training samples: {len(y_train):,}")
    print(f"Test samples: {len(y_test):,}")
    print(f"Number of clusters: {n_clusters}")
    print(f"Number of arms: 4 (MLP, CNN, TCN, BiLSTM-Attention)")
    print("-" * 80)
    
    # Initialize and train Apollon with MorphIDS pool
    apollon = MultiArmedBanditThompsonSampling(d_in=d_in, n_arms=4, n_clusters=n_clusters, seed=seed, device=DEVICE)
    apollon.train(X_train, y_train)
    
    # Predict
    start_pred = time.time()
    predictions, selected_arms = apollon.predict(X_test)
    pred_time = time.time() - start_pred
    print(f"\n    Prediction time: {pred_time:.4f} seconds")
    
    # Get probabilities
    probabilities = apollon.predict_proba(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    auc_score = roc_auc_score(y_test, probabilities)
    
    # Arm selection statistics
    print(f"\n    Arm Selection Statistics:")
    for arm_idx, arm_name in enumerate(apollon.arm_names):
        count = np.sum(selected_arms == arm_idx)
        print(f"      {arm_name}: {count:,} ({count/len(y_test)*100:.2f}%)")
    
    # Save model
    if save_path:
        joblib.dump(apollon, save_path)
        print(f"\n💾  Model saved: {save_path}")
    
    print(f"\n📊  Apollon Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:       {accuracy*100:>6.2f}%")
    print(f"    Detection Rate: {recall*100:>6.2f}%")
    print(f"    F1 Score:       {f1*100:>6.2f}%")
    print(f"    AUC-ROC:        {auc_score*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(y_test, predictions)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(y_test, predictions, 
                                target_names=['Benign', 'Attack'], digits=4))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'detection_rate': recall,
        'f1': f1,
        'auc': auc_score,
        'predictions': predictions,
        'probabilities': probabilities,
        'labels': y_test,
        'train_time': apollon.training_time,
        'model': apollon,
        'selected_arms': selected_arms
    }


# =============================================================================
# PYTORCH DATASETS AND DATALOADERS
# =============================================================================
print("\n" + "="*80)
print("CREATING PYTORCH DATASETS")
print("="*80)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train_np)
X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.LongTensor(y_val_np)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.LongTensor(y_test_np)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True if torch.cuda.is_available() else False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True if torch.cuda.is_available() else False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True if torch.cuda.is_available() else False)

# Wrap dataloaders for TPU if available
if USE_XLA and xmp is not None:
    import torch_xla.distributed.parallel_loader as pl
    train_loader = pl.MpDeviceLoader(train_loader, DEVICE)
    val_loader = pl.MpDeviceLoader(val_loader, DEVICE)
    test_loader = pl.MpDeviceLoader(test_loader, DEVICE)
    print(f"\n🚀 TPU ParallelLoader enabled")

print(f"\nDataLoader configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Train batches: {len(train_loader):>6,}")
print(f"  Val batches:   {len(val_loader):>6,}")
print(f"  Test batches:  {len(test_loader):>6,}")

# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_np), y=y_train_np)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f"\nClass weights: {class_weights}")


# =============================================================================
# MODEL TRAINING LOOP (MorphIDS Deep Learning Pool)
# =============================================================================
print("\n" + "="*80)
print("MODEL POOL TRAINING (MorphIDS Deep Learning)")
print("="*80)

input_dim = X_train_scaled.shape[1]
output_dim = 2  # Binary classification

results = {}
trained_models = {}
training_histories = {}

# MLP Model (Deep MLP from MorphIDS)
if MODELS_TO_TRAIN.get('mlp_model', False):
    model = MLPDropout(d_in=input_dim, d_out=output_dim).to(DEVICE)
    model, train_hist, val_hist, train_time = train_pytorch_model(
        'MLP (Deep Multi-Layer Perceptron)', model, train_loader, val_loader, DEVICE, class_weights_tensor, epochs=EPOCHS
    )
    trained_models['MLP'] = model
    training_histories['MLP'] = {'train': train_hist, 'val': val_hist}
    
    # Evaluate
    eval_results = evaluate_model(model, test_loader, DEVICE)
    eval_results['train_time'] = train_time
    results['MLP'] = eval_results
    
    # Save model
    model_path = os.path.join(OUT_DIR, 'mlp_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"\n💾  Model saved: {model_path}")
    
    # Print results
    print(f"\n📊  MLP Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:       {eval_results['accuracy']*100:>6.2f}%")
    print(f"    Detection Rate: {eval_results['recall']*100:>6.2f}%")
    print(f"    F1 Score:       {eval_results['f1']*100:>6.2f}%")
    print(f"    AUC-ROC:        {eval_results['auc']*100:>6.2f}%")
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'MLP', os.path.join(PLOT_DIR, 'mlp_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'MLP', os.path.join(PLOT_DIR, 'mlp_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'MLP', os.path.join(PLOT_DIR, 'mlp_pr_curve.png'))

# CNN Model (1D CNN from MorphIDS)
if MODELS_TO_TRAIN.get('cnn_model', False):
    model = DeepCNN(d_in=input_dim, d_out=output_dim).to(DEVICE)
    model, train_hist, val_hist, train_time = train_pytorch_model(
        'CNN (1D Convolutional Neural Network)', model, train_loader, val_loader, DEVICE, class_weights_tensor, epochs=EPOCHS
    )
    trained_models['CNN'] = model
    training_histories['CNN'] = {'train': train_hist, 'val': val_hist}
    
    # Evaluate
    eval_results = evaluate_model(model, test_loader, DEVICE)
    eval_results['train_time'] = train_time
    results['CNN'] = eval_results
    
    # Save model
    model_path = os.path.join(OUT_DIR, 'cnn_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"\n💾  Model saved: {model_path}")
    
    # Print results
    print(f"\n📊  CNN Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:       {eval_results['accuracy']*100:>6.2f}%")
    print(f"    Detection Rate: {eval_results['recall']*100:>6.2f}%")
    print(f"    F1 Score:       {eval_results['f1']*100:>6.2f}%")
    print(f"    AUC-ROC:        {eval_results['auc']*100:>6.2f}%")
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'CNN', os.path.join(PLOT_DIR, 'cnn_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'CNN', os.path.join(PLOT_DIR, 'cnn_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'CNN', os.path.join(PLOT_DIR, 'cnn_pr_curve.png'))

# TCN Model (Temporal CNN from MorphIDS)
if MODELS_TO_TRAIN.get('tcn_model', False):
    model = DeepTCN(d_in=input_dim, d_out=output_dim).to(DEVICE)
    model, train_hist, val_hist, train_time = train_pytorch_model(
        'TCN (Temporal Convolutional Network)', model, train_loader, val_loader, DEVICE, class_weights_tensor, epochs=EPOCHS
    )
    trained_models['TCN'] = model
    training_histories['TCN'] = {'train': train_hist, 'val': val_hist}
    
    # Evaluate
    eval_results = evaluate_model(model, test_loader, DEVICE)
    eval_results['train_time'] = train_time
    results['TCN'] = eval_results
    
    # Save model
    model_path = os.path.join(OUT_DIR, 'tcn_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"\n💾  Model saved: {model_path}")
    
    # Print results
    print(f"\n📊  TCN Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:       {eval_results['accuracy']*100:>6.2f}%")
    print(f"    Detection Rate: {eval_results['recall']*100:>6.2f}%")
    print(f"    F1 Score:       {eval_results['f1']*100:>6.2f}%")
    print(f"    AUC-ROC:        {eval_results['auc']*100:>6.2f}%")
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'TCN', os.path.join(PLOT_DIR, 'tcn_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'TCN', os.path.join(PLOT_DIR, 'tcn_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'TCN', os.path.join(PLOT_DIR, 'tcn_pr_curve.png'))

# BiLSTM-Attention Model (from MorphIDS)
if MODELS_TO_TRAIN.get('bilstm_model', False):
    model = BiLSTMAttention(d_in=input_dim, n_classes=output_dim).to(DEVICE)
    model, train_hist, val_hist, train_time = train_pytorch_model(
        'BiLSTM-Attention', model, train_loader, val_loader, DEVICE, class_weights_tensor, epochs=EPOCHS
    )
    trained_models['BiLSTM'] = model
    training_histories['BiLSTM'] = {'train': train_hist, 'val': val_hist}
    
    # Evaluate
    eval_results = evaluate_model(model, test_loader, DEVICE)
    eval_results['train_time'] = train_time
    results['BiLSTM'] = eval_results
    
    # Save model
    model_path = os.path.join(OUT_DIR, 'bilstm_model.pth')
    torch.save(model.state_dict(), model_path)
    print(f"\n💾  Model saved: {model_path}")
    
    # Print results
    print(f"\n📊  BiLSTM-Attention Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:       {eval_results['accuracy']*100:>6.2f}%")
    print(f"    Detection Rate: {eval_results['recall']*100:>6.2f}%")
    print(f"    F1 Score:       {eval_results['f1']*100:>6.2f}%")
    print(f"    AUC-ROC:        {eval_results['auc']*100:>6.2f}%")
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'BiLSTM-Attention', os.path.join(PLOT_DIR, 'bilstm_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'BiLSTM-Attention', os.path.join(PLOT_DIR, 'bilstm_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'BiLSTM-Attention', os.path.join(PLOT_DIR, 'bilstm_pr_curve.png'))

# Apollon Model (MAB with MorphIDS Deep Learning Pool)
if MODELS_TO_TRAIN.get('apollon_model', False):
    eval_results = train_apollon_model(
        X_train_scaled, y_train_np, X_test_scaled, y_test_np,
        d_in=input_dim,
        n_clusters=2,
        save_path=os.path.join(OUT_DIR, 'apollon_model.pkl')
    )
    results['Apollon'] = eval_results
    trained_models['Apollon'] = eval_results['model']
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'Apollon', os.path.join(PLOT_DIR, 'apollon_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'Apollon', os.path.join(PLOT_DIR, 'apollon_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'Apollon', os.path.join(PLOT_DIR, 'apollon_pr_curve.png'))

# =============================================================================
# VISUALIZATION SUMMARY
# =============================================================================
if results:
    print("\n" + "="*80)
    print("GENERATING COMPARISON VISUALIZATIONS")
    print("="*80)
    
    # Create comparison plots
    plot_metrics_comparison(results, os.path.join(PLOT_DIR, 'model_comparison.png'))
    plot_model_summary(results, os.path.join(PLOT_DIR, 'model_summary.png'))
    
    print(f"\n✅ All plots saved to: {PLOT_DIR}")
    print(f"   - Individual confusion matrices ({len(results)} models)")
    print(f"   - Model comparison plot")
    print(f"   - Comprehensive model summary plot")

# =============================================================================
# FINAL SUMMARY (TABLE 2 FORMAT - MorphIDS Pool)
# =============================================================================
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)

if results:
    print("\n" + "="*80)
    print("Table 2: Results of MorphIDS Deep Learning Pool + Apollon")
    print("         on the CIC-IDS-2017 dataset.")
    print("="*80)
    
    # Print header matching Table 2 format
    print(f"\n{'Metrics':<15}", end="")
    print("CIC-IDS-2017 (MorphIDS Pool)")
    print("-" * 100)
    
    # Column headers - Updated for MorphIDS models
    model_order = ['MLP', 'CNN', 'TCN', 'BiLSTM', 'Apollon']
    available_models = [m for m in model_order if m in results]
    
    print(f"{'Metrics':<15}", end="")
    for model in available_models:
        print(f"{model:>15}", end="")
    print()
    print("-" * 100)
    
    # Accuracy row
    print(f"{'Accuracy':<15}", end="")
    for model in available_models:
        print(f"{results[model]['accuracy']:>15.4f}", end="")
    print()
    
    # Detection rate row (Recall)
    print(f"{'Detection rate':<15}", end="")
    for model in available_models:
        print(f"{results[model]['recall']:>15.4f}", end="")
    print()
    
    # F1 row
    print(f"{'F1':<15}", end="")
    for model in available_models:
        print(f"{results[model]['f1']:>15.4f}", end="")
    print()
    
    # AUC row
    print(f"{'AUC':<15}", end="")
    for model in available_models:
        print(f"{results[model]['auc']:>15.4f}", end="")
    print()
    
    print("-" * 100)
    
    # Extended comparison table
    print("\n\nDetailed Model Comparison (MorphIDS Deep Learning Pool):")
    print("-" * 110)
    print(f"{'Model':<20} {'Accuracy':<12} {'Detection Rate':<15} {'F1 Score':<12} {'AUC-ROC':<12} {'Train Time':<12}")
    print("-" * 110)
    
    for model_name in available_models:
        res = results[model_name]
        train_time = f"{res.get('train_time', 0):.2f}s"
        print(f"{model_name:<20} "
              f"{res['accuracy']*100:>10.2f}% "
              f"{res['recall']*100:>13.2f}% "
              f"{res['f1']*100:>10.2f}% "
              f"{res['auc']*100:>10.2f}% "
              f"{train_time:>10}")
    
    print("-" * 110)
    
    # Find best model by F1
    best_model_name = max(results, key=lambda k: results[k]['f1'])
    best_f1 = results[best_model_name]['f1']
    print(f"\n🏆 Best Model by F1 Score: {best_model_name} (F1: {best_f1*100:.2f}%)")
    
    # Find best model by Accuracy
    best_acc_model = max(results, key=lambda k: results[k]['accuracy'])
    best_acc = results[best_acc_model]['accuracy']
    print(f"🏆 Best Model by Accuracy: {best_acc_model} (Accuracy: {best_acc*100:.2f}%)")
    
    print(f"\n✅ All models saved to: {OUT_DIR}")
else:
    print("\n⚠️ No models were trained. Check MODELS_TO_TRAIN configuration.")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
if USE_XLA:
    print(f"🚀 Hardware: TPU (torch_xla)")
elif torch.cuda.is_available():
    print(f"🚀 Hardware: GPU (CUDA)")
else:
    print(f"🔧 Hardware: CPU")
print("="*80 + "\n")


HARDWARE DETECTION
PyTorch: 2.6.0+cu124
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
⚠️ TPU not available (torch_xla not installed)
   For Kaggle TPU: Runtime → Change runtime type → TPU

✅ CUDA Available
   GPU Count: 1
   GPU 0: Tesla P100-PCIE-16GB (17.06 GB)

🎯 Using CUDA: cuda


DATA LOADING AND PREPROCESSING
Loading 7 CSV files from /kaggle/input/cic-ids2017/MachineLearningCVE...
  [1/7] Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
      Initial shape: (225745, 79)
      Memory usage: 147.66 MB
  [2/7] Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
      Initial shape: (286467, 79)
      Memory usage: 187.99 MB
  [3/7] Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
      Initial shape: (191033, 79)
      Memory usage: 125.15 MB
  [4/7] Loading: Monday-WorkingHours.pcap_ISCX.csv
      Initial shape: (529918, 79)
      Memory usage: 347.19 MB
  [5/7] Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
      Initial shape: (288602,

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_20/3981017192.py:1006: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
/tmp/ipykernel_20/3981017192.py:885: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    Epoch   1/100 - Loss: 0.1411 - Acc: 96.14% - Val Loss: 0.0501 - Val Acc: 98.67% ✓
    Epoch   2/100 - Loss: 0.0520 - Acc: 98.24% - Val Loss: 0.0363 - Val Acc: 98.91% ✓
    Epoch   3/100 - Loss: 0.0415 - Acc: 98.62% - Val Loss: 0.0315 - Val Acc: 99.07% ✓
    Epoch   4/100 - Loss: 0.0372 - Acc: 98.78% - Val Loss: 0.0276 - Val Acc: 99.25% ✓
    Epoch   5/100 - Loss: 0.0334 - Acc: 98.93% - Val Loss: 0.0262 - Val Acc: 99.20%
    Epoch   6/100 - Loss: 0.0320 - Acc: 98.98% - Val Loss: 0.0236 - Val Acc: 99.38% ✓
    Epoch   7/100 - Loss: 0.0301 - Acc: 99.06% - Val Loss: 0.0224 - Val Acc: 99.30%
    Epoch   8/100 - Loss: 0.0297 - Acc: 99.09% - Val Loss: 0.0278 - Val Acc: 99.29%
    Epoch   9/100 - Loss: 0.0287 - Acc: 99.13% - Val Loss: 0.0240 - Val Acc: 98.88%
    Epoch  10/100 - Loss: 0.0278 - Acc: 99.15% - Val Loss: 0.0226 - Val Acc: 99.02%
    Epoch  11/100 - Loss: 0.0285 - Acc: 99.13% - Val Loss: 0.0216 - Val Acc: 99.20%
    Epoch  12/100 - Loss: 0.0279 - Acc: 99.16% - Val Loss: 0.0199 

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_20/3981017192.py:1006: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
/tmp/ipykernel_20/3981017192.py:885: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    Epoch   1/100 - Loss: 0.2068 - Acc: 93.27% - Val Loss: 0.0573 - Val Acc: 97.02% ✓
    Epoch   2/100 - Loss: 0.0473 - Acc: 98.30% - Val Loss: 0.0549 - Val Acc: 99.12% ✓
    Epoch   3/100 - Loss: 0.0396 - Acc: 98.59% - Val Loss: 0.0374 - Val Acc: 98.46%
    Epoch   4/100 - Loss: 0.0382 - Acc: 98.65% - Val Loss: 0.0318 - Val Acc: 98.46%
    Epoch   5/100 - Loss: 0.0344 - Acc: 98.81% - Val Loss: 0.0340 - Val Acc: 98.20%
    Epoch   6/100 - Loss: 0.0315 - Acc: 98.93% - Val Loss: 0.0263 - Val Acc: 99.13% ✓
    Epoch   7/100 - Loss: 0.0288 - Acc: 99.06% - Val Loss: 0.0445 - Val Acc: 98.03%
    Epoch   8/100 - Loss: 0.0297 - Acc: 99.03% - Val Loss: 0.0268 - Val Acc: 99.24% ✓
    Epoch   9/100 - Loss: 0.0283 - Acc: 99.08% - Val Loss: 0.0231 - Val Acc: 99.05%
    Epoch  10/100 - Loss: 0.0265 - Acc: 99.16% - Val Loss: 0.0525 - Val Acc: 99.24%
    Epoch  11/100 - Loss: 0.0262 - Acc: 99.18% - Val Loss: 0.0222 - Val Acc: 99.40% ✓
    Epoch  12/100 - Loss: 0.0249 - Acc: 99.22% - Val Loss: 0.0329 

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_20/3981017192.py:1006: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
/tmp/ipykernel_20/3981017192.py:885: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    Epoch   1/100 - Loss: 0.1896 - Acc: 91.08% - Val Loss: 0.0562 - Val Acc: 98.24% ✓
    Epoch   2/100 - Loss: 0.0495 - Acc: 98.13% - Val Loss: 0.0415 - Val Acc: 98.03%
    Epoch   3/100 - Loss: 0.0412 - Acc: 98.54% - Val Loss: 0.0412 - Val Acc: 99.32% ✓
    Epoch   4/100 - Loss: 0.0359 - Acc: 98.77% - Val Loss: 0.0360 - Val Acc: 98.27%
    Epoch   5/100 - Loss: 0.0338 - Acc: 98.87% - Val Loss: 0.0311 - Val Acc: 98.47%
    Epoch   6/100 - Loss: 0.0310 - Acc: 98.99% - Val Loss: 0.0285 - Val Acc: 98.96%
    Epoch   7/100 - Loss: 0.0292 - Acc: 99.08% - Val Loss: 0.0257 - Val Acc: 99.22%
    Epoch   8/100 - Loss: 0.0280 - Acc: 99.11% - Val Loss: 0.0249 - Val Acc: 99.40% ✓
    Epoch   9/100 - Loss: 0.0270 - Acc: 99.15% - Val Loss: 0.0235 - Val Acc: 99.32%
    Epoch  10/100 - Loss: 0.0257 - Acc: 99.22% - Val Loss: 0.0258 - Val Acc: 98.84%
    Epoch  11/100 - Loss: 0.0253 - Acc: 99.23% - Val Loss: 0.0230 - Val Acc: 99.45% ✓
    Epoch  12/100 - Loss: 0.0235 - Acc: 99.31% - Val Loss: 0.0214 - 

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_20/3981017192.py:1006: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (torch.cuda.is_available() and not USE_XLA) else None
/tmp/ipykernel_20/3981017192.py:885: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    Epoch   1/100 - Loss: 0.0976 - Acc: 97.18% - Val Loss: 0.0425 - Val Acc: 98.09% ✓
    Epoch   2/100 - Loss: 0.0474 - Acc: 98.42% - Val Loss: 0.0318 - Val Acc: 98.62% ✓
    Epoch   3/100 - Loss: 0.0397 - Acc: 98.64% - Val Loss: 0.0298 - Val Acc: 98.90% ✓
    Epoch   4/100 - Loss: 0.0359 - Acc: 98.79% - Val Loss: 0.0243 - Val Acc: 99.22% ✓
    Epoch   5/100 - Loss: 0.0366 - Acc: 98.71% - Val Loss: 0.0420 - Val Acc: 97.94%
    Epoch   6/100 - Loss: 0.0341 - Acc: 98.82% - Val Loss: 0.0244 - Val Acc: 99.08%
    Epoch   7/100 - Loss: 0.0324 - Acc: 98.89% - Val Loss: 0.0252 - Val Acc: 99.39% ✓
    Epoch   8/100 - Loss: 0.0310 - Acc: 98.98% - Val Loss: 0.0237 - Val Acc: 99.39% ✓
    Epoch   9/100 - Loss: 0.0307 - Acc: 98.99% - Val Loss: 0.0244 - Val Acc: 99.29%
    Epoch  10/100 - Loss: 0.0318 - Acc: 98.93% - Val Loss: 0.0242 - Val Acc: 98.94%
    Epoch  11/100 - Loss: 0.0289 - Acc: 99.05% - Val Loss: 0.0242 - Val Acc: 99.31%
    Epoch  12/100 - Loss: 0.0295 - Acc: 99.04% - Val Loss: 0.022